<div align='center'>

# **Práctica 2 — Minería de Textos y Análisis de Temas**

**Máster Universitario en Inteligencia Artificial**  
**Asignatura:** Minería de Textos y Modelos de Lenguaje

<br/>
Autores:
- Cristian Rubio Barato — <a href='mailto:Cristian.Rubio@alu.uclm.es'>Cristian.Rubio@alu.uclm.es</a>  
- José Lara Navarro — <a href='mailto:Jose.Lara3@alu.uclm.es'>Jose.Lara3@alu.uclm.es</a>

</div>

<a id='indice'></a>

**Índice**

- [1. Carga, preprocesado y limpieza del conjunto de datos](#sec1)
- [2. Vectorización del corpus](#sec2)
- [3. Declaración de uso responsable de Inteligencia Artificial](#sec3)

<a id='sec1'></a>
## 1. Carga, preprocesado y limpieza del conjunto de datos

Cargamos el dataset, extraemos la columna `cuerpo` como corpus y aplicamos el pipeline canónico:

1. **Limpieza básica**: minúsculas, eliminación de URLs, caracteres no alfabéticos y normalización de espacios.
2. **Lematización con spaCy**: procesamiento por lotes con barra de progreso, filtrando stopwords, puntuación, espacios y tokens de longitud 1. Se usa `n_process=1` para evitar el error de memoria que produce el multiprocesamiento en Windows con modelos grandes.

El resultado se guarda en `corpus_lemas.pkl` para no repetir la lematización en ejecuciones posteriores.

In [2]:
import pandas as pd
import re
import pickle
import os
import spacy
from tqdm import tqdm

# ── 1. Extracción del corpus ───────────────────────────────────────────────────
df = pd.read_csv("data_larazon_publico_v2.csv")
print(f"Dimensiones del dataset: {df.shape}")

corpus = df["cuerpo"].dropna().reset_index(drop=True)
print(f"Documentos en el corpus (tras eliminar NaN): {len(corpus)}")
print(f"\nEjemplo (primeros 200 caracteres):\n  {corpus.iloc[0][:200]}...")


# ── 2. Limpieza básica ─────────────────────────────────────────────────────────
def limpiar_texto(texto):
    texto = str(texto).lower()
    texto = re.sub(r"http\S+|www\S+", "", texto)  # Eliminar URLs
    texto = re.sub(r"[^a-záéíóúüñ\s]", " ", texto)  # Solo caracteres alfabéticos
    texto = re.sub(r"\s+", " ", texto).strip()  # Normalizar espacios
    return texto


corpus_limpio = corpus.apply(limpiar_texto)
print("\nEjemplo original vs. limpio:")
print(f"  Original : {corpus.iloc[0][:100]}")
print(f"  Limpio   : {corpus_limpio.iloc[0][:100]}")

# ── 3. Lematización con spaCy ─────────────────────────────────────────────────
# n_process=1 evita el ValueError de memoria que ocurre con n_process>1 en Windows.
# batch_size=128 reduce el pico de RAM por lote manteniendo un rendimiento razonable.
PICKLE_PATH = "corpus_lemas.pkl"


def lematizar_corpus(corpus_serie, batch_size=128):
    nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
    textos = corpus_serie.tolist()
    salida = []
    for doc in tqdm(
        nlp.pipe(textos, batch_size=batch_size, n_process=1),
        total=len(textos),
        desc="Lematizando",
    ):
        tokens = [
            t.lemma_
            for t in doc
            if not t.is_stop
            and not t.is_punct
            and not t.is_space
            and t.is_alpha
            and len(t.lemma_) > 1
        ]
        salida.append(" ".join(tokens))
    return pd.Series(salida, index=corpus_serie.index)


if os.path.exists(PICKLE_PATH):
    print(f"\nCargando corpus lematizado desde '{PICKLE_PATH}'...")
    with open(PICKLE_PATH, "rb") as f:
        lemas = pickle.load(f)
    print("Corpus cargado correctamente.")
else:
    print("\nIniciando lematización (esto puede tardar varios minutos)...")
    lemas = lematizar_corpus(corpus_limpio)
    with open(PICKLE_PATH, "wb") as f:
        pickle.dump(lemas, f)
    print(f"Corpus guardado en '{PICKLE_PATH}'.")

# ── Verificación ──────────────────────────────────────────────────────────────
print("\nEjemplos de texto limpio vs. lematizado:\n")
for i in [0, 1, 2]:
    print(f"  Limpio     : {corpus_limpio.iloc[i][:100]}")
    print(f"  Lematizado : {lemas.iloc[i][:100]}")
    print()

print("Corpus lematizado (Serie de pandas):")
lemas

Dimensiones del dataset: (58424, 4)
Documentos en el corpus (tras eliminar NaN): 58424

Ejemplo (primeros 200 caracteres):
  dos semanas después de su puesta de largo y presentación en sociedad, el primer submarino s-80 para la armada, el s-81 "isaac peral", ha entrado hoy en el agua tras una delicada y larga maniobra que s...

Ejemplo original vs. limpio:
  Original : dos semanas después de su puesta de largo y presentación en sociedad, el primer submarino s-80 para 
  Limpio   : dos semanas después de su puesta de largo y presentación en sociedad el primer submarino s para la a

Iniciando lematización (esto puede tardar varios minutos)...


Lematizando: 100%|██████████| 58424/58424 [36:41<00:00, 26.54it/s]  


Corpus guardado en 'corpus_lemas.pkl'.

Ejemplos de texto limpio vs. lematizado:

  Limpio     : dos semanas después de su puesta de largo y presentación en sociedad el primer submarino s para la a
  Lematizado : semana puesta presentación sociedad submarino armada isaac peral entrar agua delicado largo maniobra

  Limpio     : este viernes el presidente del gobierno pedro sánchez y las cuatro vicepresidentas carmen calvo nadi
  Lematizado : viernes presidente gobierno pedro sánchez vicepresidenta carmen calvo nadia calviño yolanda díaz ter

  Limpio     : el ministro del interior fernando grande marlaska y el alcalde de guadalajara alberto rojo blas han 
  Lematizado : ministro interior fernando marlaska alcalde guadalajara alberto rojo bla suscribir convenio colabora

Corpus lematizado (Serie de pandas):


0        semana puesta presentación sociedad submarino ...
1        viernes presidente gobierno pedro sánchez vice...
2        ministro interior fernando marlaska alcalde gu...
3        duro familia olivia anna gimeno zimmerman cump...
4        quedar preso eta recibir beneficio traslado pa...
                               ...                        
58419    comisión europea iniciar procedimiento infracc...
58420    pleno asamblea madrid aprobar jueves proyecto ...
58421    comisión investigación parlamentario accidente...
58422    erc pdecat calificar jueves inaceptable grabac...
58423    junta portavoz congreso acordar oposición pp a...
Length: 58424, dtype: str

<a id='sec2'></a>
## 2. Vectorización del corpus

Transformamos el corpus lematizado en una representación numérica mediante **Bag of Words (BoW)** con `CountVectorizer` de scikit-learn.

Con ~58.000 documentos, `min_df` absoluto tiene mucho más margen que en la P1: un valor de 20 ya filtra términos presentes en menos del 0.03% del corpus. Exploramos varias configuraciones antes de fijar la definitiva.

In [3]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

# ── Exploración de configuraciones ────────────────────────────────────────────
configuraciones = [
    {"ngram_range": (1, 1), "min_df": 5, "max_df": 0.95, "max_features": None},
    {"ngram_range": (1, 1), "min_df": 10, "max_df": 0.90, "max_features": None},
    {"ngram_range": (1, 2), "min_df": 10, "max_df": 0.90, "max_features": None},
    {"ngram_range": (1, 2), "min_df": 20, "max_df": 0.90, "max_features": None},
    {"ngram_range": (1, 2), "min_df": 20, "max_df": 0.90, "max_features": 10000},
    {"ngram_range": (1, 2), "min_df": 50, "max_df": 0.85, "max_features": 10000},
    {"ngram_range": (1, 2), "min_df": 100, "max_df": 0.85, "max_features": 10000},
    {"ngram_range": (1, 2), "min_df": 0.005, "max_df": 0.80, "max_features": 10000},
]

print(
    f"{'Config':<8} {'n-gramas':<12} {'min_df':<10} {'max_df':<8} {'max_feat':<12} {'Terminos':<10}"
)
print("-" * 64)
for i, config in enumerate(configuraciones):
    vec_tmp = CountVectorizer(**config)
    dtm_tmp = vec_tmp.fit_transform(lemas)
    mf = config["max_features"]
    print(
        f"  {i + 1:<6} {str(config['ngram_range']):<12} {str(config['min_df']):<10} "
        f"{config['max_df']:<8} {str(mf):<12} {dtm_tmp.shape[1]:<10}"
    )

print("\nTop 10 términos más frecuentes por configuración:\n")
for i, config in enumerate(configuraciones):
    vec_tmp = CountVectorizer(**config)
    dtm_tmp = vec_tmp.fit_transform(lemas)
    sumas = np.asarray(dtm_tmp.sum(axis=0)).flatten()
    top_idx = sumas.argsort()[::-1][:10]
    top_terms = [vec_tmp.get_feature_names_out()[j] for j in top_idx]
    print(f"  Config {i + 1}: {top_terms}")

# ── Vectorización final ────────────────────────────────────────────────────────
# Elegimos la configuración 5: bigramas, min_df=20, max_df=0.90, max_features=10000.
# Ofrece el mejor equilibrio entre riqueza de vocabulario y eliminación de ruido.
print("\n" + "=" * 60)
print("=== VECTORIZACIÓN FINAL — BAG OF WORDS ===")
print("=" * 60)

vectorizador = CountVectorizer(
    ngram_range=(1, 2), min_df=20, max_df=0.90, max_features=10000
)
dtm_sparse = vectorizador.fit_transform(lemas)
feature_names = vectorizador.get_feature_names_out()

# 1) DTM — mantenemos formato sparse para no saturar RAM; lo mostramos como DataFrame
dtm_df = pd.DataFrame.sparse.from_spmatrix(dtm_sparse, columns=feature_names)
print("\n1) Matriz DTM (Document-Term Matrix):")
print(dtm_df)

# 2) Tamaño
print(f"\n2) Tamaño final del DTM:")
print(f"   Documentos : {dtm_sparse.shape[0]}")
print(f"   Términos   : {dtm_sparse.shape[1]}")

# 3) Top 20 términos más frecuentes
sumas = np.asarray(dtm_sparse.sum(axis=0)).flatten()
top20_idx = sumas.argsort()[::-1][:20]
top20 = pd.Series(sumas[top20_idx], index=feature_names[top20_idx])
print("\n3) Top 20 términos más frecuentes (BoW):")
print(top20.to_string())

Config   n-gramas     min_df     max_df   max_feat     Terminos  
----------------------------------------------------------------
  1      (1, 1)       5          0.95     None         41522     
  2      (1, 1)       10         0.9      None         28276     
  3      (1, 2)       10         0.9      None         199671    
  4      (1, 2)       20         0.9      None         88064     
  5      (1, 2)       20         0.9      10000        10000     
  6      (1, 2)       50         0.85     10000        10000     
  7      (1, 2)       100        0.85     10000        10000     
  8      (1, 2)       0.005      0.8      10000        10000     

Top 10 términos más frecuentes por configuración:

  Config 1: ['él', 'gobierno', 'año', 'persona', 'partido', 'hora', 'españa', 'mantener', 'caso', 'madrid']
  Config 2: ['él', 'gobierno', 'año', 'persona', 'partido', 'hora', 'españa', 'mantener', 'caso', 'madrid']
  Config 3: ['él', 'gobierno', 'año', 'persona', 'partido', 'hora', 'espa

<a id='sec3'></a>
## 3. Declaración de uso responsable de Inteligencia Artificial

En la elaboración de esta práctica, se ha utilizado la IA como herramienta de apoyo. Su uso se ha limitado a los siguientes aspectos:

- Mejora de la presentación visual de resultados, incluyendo el formateo de tablas y la disposición de los bloques de código.
- Corrección de errores puntuales de código, como problemas de compatibilidad entre librerías o ajustes sintácticos menores.
- Asistencia en la redacción de comentarios de código y textos explicativos, partiendo siempre de las ideas y el análisis previo de los autores.

En ningún caso se ha delegado en la herramienta la toma de decisiones metodológicas, la elección de parámetros, la interpretación de los resultados ni la elaboración de las conclusiones. Todo el criterio analítico, las reflexiones críticas y las valoraciones recogidas en este documento son responsabilidad exclusiva de los autores.